# DEKM 2021 paper-era RCV1 seed-42 audit

This immutable diagnostic runs the 2021 repository contract for RCV1: unit-L2 text input, uniform initialization, `np.linalg.eig`, 256-row refinement batches, and the released 0.5% stopping threshold. It tests the DEKM row, not DEKM_F, and changes no other factor.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
assert gpus, 'Enable a GPU accelerator in Kaggle notebook settings.'


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/mvdec_fpmax')
subprocess.run(['rm', '-rf', str(REPO)], check=True)
subprocess.run([
    'git', 'clone', '--branch', 'dev', '--single-branch',
    'https://github.com/vdat-18/mvdec_fpmax.git', str(REPO),
], check=True)
EXPECTED_REVISION = '1c75e0de7aaea5b6b5974d56ccc701d795909677'
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'gdown'], check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual == EXPECTED_REVISION, actual
print('Repository revision:', actual)

subprocess.run([
    sys.executable, str(REPO / 'scripts/fetch_dekm_release.py'),
    '--datasets', 'RCV1',
], cwd=REPO, check=True)
from sklearn.datasets import fetch_rcv1
fetch_rcv1(download_if_missing=True)
print('RCV1 source and official split indices are ready.')


In [ ]:
OUTPUT = Path('/kaggle/working/dekm-2021-rcv1-paper-era-seed42')
RUN_ROOT = OUTPUT / 'runs'
command = [
    sys.executable, '-m', 'representation_learning.DEKM_dense', 'RCV1',
    '--protocol', 'release-2021',
    '--seed', '42', '--runs', '1', '--require-gpu',
    '--dataset-root', str(REPO / 'external_repos/DEKM/datasets'),
    '--output-dir', str(RUN_ROOT),
]
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import json
import pandas as pd
from pipeline.external_metrics import compute_external_metrics

PROTOCOL = 'dekm_2021_release_seeded_modern_runtime_v1'
protocol_root = RUN_ROOT / 'rcv1' / PROTOCOL
config_roots = [path for path in protocol_root.iterdir() if path.is_dir()]
assert len(config_roots) == 1, config_roots
run_dir = config_roots[0] / 'seed_42'
manifest_path = run_dir / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['status'] == 'complete'
assert manifest['method'] == 'DEKM'
assert manifest['protocol_id'] == PROTOCOL
assert manifest['config']['refinement_batch_size'] == 256
assert manifest['config']['assignment_change_tolerance'] == 0.005
assert manifest['config']['input_normalization'] == 'release_l2_unit_norm'
assert manifest['official_repository_paper_era_revision'] == '129b7d08b545f02d920ccff0948ec44b654b3158'
assert manifest['official_repository_paper_era_dense_sha256'] == '1298eff9d4de23f75dccc8a341945f98e908521cf100c440b43a6521d78e5185'

assignments = pd.read_csv(run_dir / 'assignments.csv')
metrics = compute_external_metrics(
    assignments['true_label'].to_numpy(),
    assignments['cluster'].to_numpy(),
    n_clusters=4,
)
assert abs(metrics.acc - manifest['metrics']['acc']) < 1e-12
assert abs(metrics.nmi - manifest['metrics']['nmi']) < 1e-12

PAPER_DEKM = {'acc': 0.6715, 'nmi': 0.4618}
PAPER_DEKM_F = {'acc': 0.6212, 'nmi': 0.3598}
row = {'dataset': 'RCV1', 'seed': 42, 'method': 'DEKM', 'protocol_id': PROTOCOL, 'source_sha256': manifest['source_sha256'], 'acc': float(metrics.acc), 'nmi': float(metrics.nmi), 'fit_time_seconds': float(manifest['fit_time_seconds']), 'steps_completed': int(manifest['steps_completed']), 'refreshes_completed': int(manifest['refreshes_completed']), 'stop_reason': manifest['stop_reason'], 'paper_dekm_acc': PAPER_DEKM['acc'], 'paper_dekm_nmi': PAPER_DEKM['nmi'], 'paper_dekm_acc_delta_pp': float(100 * (metrics.acc - PAPER_DEKM['acc'])), 'paper_dekm_nmi_delta_pp': float(100 * (metrics.nmi - PAPER_DEKM['nmi'])), 'paper_dekm_f_acc': PAPER_DEKM_F['acc'], 'paper_dekm_f_nmi': PAPER_DEKM_F['nmi']}
diagnostic = pd.DataFrame([row])
diagnostic.to_csv(OUTPUT / 'diagnostic.csv', index=False)
(OUTPUT / 'diagnostic.json').write_text(json.dumps(row, indent=2), encoding='utf-8')
display(diagnostic)


In [ ]:
import hashlib
import shutil

zip_path = Path(shutil.make_archive(
    '/kaggle/working/dekm-2021-rcv1-paper-era-seed42',
    'zip', '/kaggle/working', 'dekm-2021-rcv1-paper-era-seed42',
))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = zip_path.with_suffix('.zip.sha256')
sha_path.write_text(f'{digest}  {zip_path.name}\n', encoding='ascii')
print('ZIP:', zip_path)
print('SHA-256:', digest)
print('SHA file:', sha_path)
